# Baseline CNN / RNN / ViT / lai trên AudioSet-40l

16 baseline có 0,7-28M tham số, huấn luyện **từ đầu** trên cùng log-mel 128 x 960 (9,6 s), cùng SpecAugment,
cùng optimizer, lịch học và tiêu chí chọn (`macro_f1` trên validation), rồi đánh giá bằng **9 metrics**.

| Họ | Mô hình |
| --- | --- |
| CNN | MobileNetV3-L (width 0,4 / 0,5 / 1,0 / 2,0), PANNs CNN10, ResNet18, EfficientNet-B2, ConvNeXt V2-T |
| RNN | CRNN (BiGRU), BiLSTM |
| ViT | DeiT-Tiny, ViT-Mini, ViT-S/16 |
| Lai | MobileViT-S, Conformer-M, Swin-T |

Tên mô hình trong bảng kết quả ghi rõ **kiến trúc**, không phải trọng số: không có checkpoint EfficientAT,
PANNs, CED hay HTS-AT nào được nạp. Cấu hình nằm ở `config/baselines.yaml`.

In [ ]:
# Chạy từ thư mục gốc của repo để import được `scripts` và `src`
import os
import sys
from pathlib import Path

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "rules.MD").is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ROOT

In [ ]:
# (1) Tên pipeline, chế độ chạy, đường dẫn đầu ra (RunPaths), logger
import torch

from src.utils.logger import setup_logger
from src.utils.paths import RunPaths

PIPELINE_NAME = "baselines"
RUN_TRAINING = False  # True: chạy thật; False: smoke test nhỏ ghi vào output/test
RESUME_RUN_ID = None  # ví dụ "baselines-2026-10-14-21-50-17" để chạy tiếp lần chạy cũ

output_dir = Path("output") if RUN_TRAINING else Path("output") / "test"
if RESUME_RUN_ID:
    run = RunPaths.from_run_id(RESUME_RUN_ID, output_dir)
else:
    run = RunPaths.new(PIPELINE_NAME if RUN_TRAINING else f"{PIPELINE_NAME}-smoke", output_dir)
run.create()

logger = setup_logger(PIPELINE_NAME, run.log_file)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("run %s | device %s | RUN_TRAINING=%s", run.run_id, device, RUN_TRAINING)

In [ ]:
# (2) Tham số cấu hình: tập baseline cần chạy (None = cả 12), số worker, model xem confusion matrix
SELECTED_MODELS = None
NUM_WORKERS = 4
CONFUSION_MODEL = None  # None: model có macro_f1 cao nhất

In [ ]:
# (3) Nạp dataset (tự tải từ Kaggle khi chưa có)
from scripts.download_dataset import ensure_dataset
from scripts.load_dataset import build_dataset
from scripts.run_baselines import read_baselines_config

dataset_config = read_baselines_config()["dataset"]
dataset_root = ensure_dataset(dataset_config["name"], Path("datasets"))
class_names = build_dataset(dataset_config["name"], "train", duration=dataset_config["duration"]).class_names
print(f"{dataset_config['name']}: {dataset_root} | {len(class_names)} lớp | clip {dataset_config['duration']} s")

## (4) Pretrain

Không nạp checkpoint pretrain: checkpoint AudioSet đã thấy nhãn của các clip này (rò rỉ) và dùng front-end
32 kHz / 64 mel khác dữ liệu 16 kHz. Cờ `pretrained` trong `config/baselines.yaml` chỉ bật được cho các kiến trúc
timm (khởi tạo ImageNet).

In [ ]:
# (5) Dựng kiến trúc và đếm tham số thực tế với đầu ra đúng số lớp
from IPython.display import Image, Markdown, display

from scripts.run_baselines import describe_baselines, markdown_table

architectures = describe_baselines(num_classes=len(class_names))
display(Markdown(markdown_table(architectures, ("model", "family", "architecture", "params_m"), digits=2)))

In [ ]:
# (6) Huấn luyện từng baseline (bỏ qua model đã xong), ghi result.csv sau mỗi model
from scripts.run_baselines import run_baselines

rows = run_baselines(
    run,
    device,
    logger,
    names=SELECTED_MODELS,
    num_workers=NUM_WORKERS,
    smoke=not RUN_TRAINING,
)

In [ ]:
# Bảng so sánh 16 baseline theo 9 metrics (xếp theo macro_f1), ghi comparison.csv
from scripts.finetune import METRIC_NAMES
from scripts.run_baselines import COMPARISON_COLUMNS, best_models, comparison_table, write_comparison

comparison_path = write_comparison(rows, run)
display(Markdown(f"Bảng so sánh (validation, xếp theo `macro_f1`): `{comparison_path}`"))
display(Markdown(markdown_table(comparison_table(rows), COMPARISON_COLUMNS)))

leaders = best_models(rows)
display(
    Markdown("Model tốt nhất theo từng metric:\n\n" + "\n".join(f"- `{m}`: **{leaders[m]}**" for m in METRIC_NAMES))
)

In [ ]:
# Confusion matrix (tỉ lệ trên lớp thật) của từng baseline, ghi vào output/figure/<run>/
from scripts.run_baselines import load_confusions, plot_baseline_confusions

figure_paths = plot_baseline_confusions(load_confusions(run), class_names, run)
shown = CONFUSION_MODEL or leaders["macro_f1"]
print(f"model: {shown} | các model có ảnh: {sorted(figure_paths)}")
display(Image(filename=str(figure_paths[shown])))